### Inference test set is tricky to simulate

- Dear New Competitors,
I spent over a week trying to make my first submission and finally succeeded after making tens of attempts.
What was frustrating is that the submission notebook threw exceptions, and one has to rely on guessing to figure out what goes wrong at inference time.

I wrote this notebook to help you simulate the test set as closely as possible during inference time, so that you can debug and test your code effectively.



### Partitioning the Columns

**Target Column:**  
`gesture`

**Common Extra Columns:**  
`row_id`, `sequence_counter`

**Rotation Columns:**  
`rot_w`, `rot_x`, `rot_y`, `rot_z`

**Accelerometer Columns:**  
`acc_x`, `acc_y`, `acc_z`

**Thermophile Columns:**  
`thm_1`, `thm_2`, `thm_3`, `thm_4`, `thm_5`

**Time-of-Flight (ToF) Columns:**  
`tof_1_v1`, `tof_1_v2`, ..., `tof_5_v64`  
<sub>(64 columns for each of the 5 sensors)</sub>

**Grouping Column:**  
`sequence_id`

**Subject Column:**  
`subject`



**Categorical Columns to Encode:**  
`sex`, `handedness`, `adult_child`

In [3]:
from pathlib import Path
import polars as pl

target_col = "gesture"
common_extra_cols = ["row_id","sequence_counter"]
rotation_cols = [ 'rot_w', 'rot_x',
       'rot_y', 'rot_z']
accelerometer_cols = ['acc_x', 'acc_y', 'acc_z']

thermophile_columns = [f"thm_{i}" for i in range(1, 6)]
tof_cols = [f"tof_{i}_v{j}" for i in range(1, 6) for j in range(64)]
grouping_col = "sequence_id"
subject_col = "subject"
heat_col = "total_heat"
# doing the fe
cats_to_be = ["sex","handedness","adult_child"]

In [4]:
#retrain the model from scratch, but first testing the model that I have
train_df = pl.read_csv('train.csv')
train_demo_df = pl.read_csv('train_demographics.csv')

### Start randomly nullifying the dfs' columns

To simulate missing data, we randomly nullify parts of the DataFrame's columns. This helps identify potential issues during inference.

In [83]:
def randomly_nullify_df(train_df,parts_count=2):
    row_no = train_df.height //parts_count
    random_indices = np.random.choice(np.arange(row_no))
    all_cols = train_df.columns
    for col in all_cols:
        b_df = train_df.with_columns(pl.when(pl.arange(0,train_df.height).is_in(random_indices)).
                                     then(None).otherwise(train_df[col]).alias(col))
    return b_df
        

In [85]:
preprocessed_demog_df = randomly_nullify_df(train_demo_df)
preprocessed_quantitative_df = randomly_nullify_df(train_df)

C:\Users\Moetasim Rady\AppData\Local\Temp\ipykernel_22280\1743824063.py:6: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  b_df = train_df.with_columns(pl.when(pl.arange(0,train_df.height).is_in(random_indices)).


In [120]:
def randomly_sample(train_df,demog_df,idx=None):
    rows_no = train_df.height//2
    subj_range = len(train_df[subject_col].unique())
    random_idx = np.random.choice(np.arange(subj_range)) if idx is None else idx
    subjc_id = demog_df[subject_col].to_list()[random_idx]
    quant_seq_df = train_df.filter(pl.col(subject_col)==subjc_id)
    demog_seq_df = demog_df.filter(pl.col(subject_col)==subjc_id)
    return quant_seq_df,demog_seq_df

##########


#### Randomly sampling from training data with null values 
We randomly sample from the training data (with nullified values) to simulate inference conditions and test the robustness of the pipeline.

In [ ]:
normal_seq,demog_seq = randomly_sample(b_df,b_demog_df)



'Forehead - pull hairline'

Next Steps
- **Implement the `predict` function** to handle the simulated test data.
- Use the randomly nullified and sampled data to ensure your pipeline is robust and can handle edge cases effectively.